In [1]:
import pandas as pd
import pyodbc
import numpy as np

## PERSON INFORMATIONS

In [ ]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER= CONNECTION;"
    "DATABASE=AdventureWorks2022;"
    "Trusted_Connection=yes;"
)

person_main = pd.read_sql("SELECT * FROM Person.Person", conn)
person_phone = pd.read_sql("SELECT * FROM Person.PersonPhone", conn)
person_email =  pd.read_sql("SELECT * FROM Person.EmailAddress", conn)


conn.close()


In [49]:
person_main["FullName"] = (
    person_main[["FirstName", "MiddleName", "LastName"]]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [51]:
person_main_copy = person_main

In [52]:
person_main = person_main_copy[["BusinessEntityID", "FullName"]]

In [54]:
phone_num_type_mapping = {
    1: "Cell",
    2: "Home",
    3: "Work"
}
person_phone["PhoneNumberType"] = person_phone["PhoneNumberTypeID"].map(phone_num_type_mapping) 

In [55]:
person_phone_copy = person_phone

In [56]:
person_phone = person_phone[["BusinessEntityID", "PhoneNumber", "PhoneNumberType"]]

In [57]:
person_email = person_email[["BusinessEntityID", "EmailAddress"]]

In [53]:
from sqlalchemy import create_engine

engine = create_engine(
    "CONNECTION"
)

person_main.to_sql(
    "persons",
    engine,
    if_exists="replace",
    schema = "staging",
    index=False
)



972

In [58]:
person_email.to_sql(
    "person_email",
    engine,
    if_exists="replace",
    schema = "staging",
    index=False
)

972

In [59]:
person_phone.to_sql(
    "person_phone_number",
    engine,
    if_exists="replace",
    schema = "staging",
    index=False
)

400

## EMPLOYEE INFORMATIONS

In [ ]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=CONNECTION;"
    "DATABASE=AdventureWorks2022;"
    "Trusted_Connection=yes;"
)

department_history =  pd.read_sql("SELECT * FROM HumanResources.EmployeeDepartmentHistory", conn)
sales_person = pd.read_sql("SELECT * FROM Sales.SalesPerson", conn)


conn.close()

In [5]:
department_history = department_history.sort_values(["BusinessEntityID", "StartDate"], ascending=[True, False])

In [6]:
department_history["row_number"] = department_history.groupby("BusinessEntityID").cumcount() + 1

In [7]:
department_history = department_history[department_history["row_number"] == 1]

In [8]:
department_shift_mapping = {
    1: "Day",
    2: "Evening",
    3: "Night"   
}


In [9]:
department_history["ShiftType"] = department_history["ShiftID"].map(department_shift_mapping)

In [10]:
department_history = department_history[["BusinessEntityID", "DepartmentID", "StartDate","ShiftType"]]

In [16]:
department_history["IsSalesPerson"] = department_history["BusinessEntityID"].isin( sales_person["BusinessEntityID"]).map({True: "Yes", False: "No"})

In [19]:
from sqlalchemy import create_engine

engine = create_engine(
    "CONNECTION"
)

department_history.to_sql(
    "employee_department_history",
    engine,
    if_exists="replace",
    schema = "staging",
    index=False
)



290

## CUSTOMER INFORMATIONS

In [ ]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=CONNECTION;"
    "DATABASE=AdventureWorks2022;"
    "Trusted_Connection=yes;"
)

customer =  pd.read_sql("SELECT * FROM Sales.Customer", conn)
store = pd.read_sql("select * from Sales.Store", conn)


conn.close()

In [8]:
customer = customer.merge(store, how="left", left_on = "StoreID", right_on = "BusinessEntityID", suffixes=("_customer", "_store"))

In [13]:
customer = customer[["CustomerID", "AccountNumber", "PersonID", "StoreID","TerritoryID","Name"]]

In [19]:
customer_type_condition =  [
    (customer["StoreID"].notna()) & (customer["PersonID"].notna()),
    (customer["PersonID"].isna()),
    (customer["StoreID"].isna())
    
]

customer_type_choices =  [
    "Flagged",
    "Store",
    "Customer"   
]

customer["CustomerType"] = np.select(customer_type_condition, customer_type_choices, default = "Unknown")

In [22]:
customer = customer.rename(columns = {"Name": "StoreName"})

In [24]:
from sqlalchemy import create_engine

engine = create_engine(
    "CONNECTION"
)

customer.to_sql(
    "customer",
    engine,
    if_exists="replace",
    schema = "staging",
    index=False
)


86

## SALES INFORMATIONS

In [ ]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=CONNECTION"
    "DATABASE=AdventureWorks2022;"
    "Trusted_Connection=yes;"
)

orders =  pd.read_sql("select * from Sales.SalesOrderHeader", conn)
order_details = pd.read_sql("select * from Sales.SalesOrderDetail", conn)
currency_rate = pd.read_sql("select * from Sales.CurrencyRate", conn)
special_offer = pd.read_sql("select  SpecialOfferID, [Type] from Sales.SpecialOffer", conn)



conn.close()

In [4]:
orders["OnlineOrder"] = orders["OnlineOrderFlag"].map({True: "Yes", False: "No"})

In [5]:
ship_method = {
    1: "XRQ - TRUCK GROUND",
    2: "ZY - EXPRESS",
    3: "OVERSEAS - DELUXE",
    4: "OVERNIGHT J-FAST",
    5: "CARGO TRANSPORT 5"
}

orders["ShipMethod"] = orders["ShipMethodID"].map(ship_method)

In [6]:
orders = orders[["SalesOrderID", "OrderDate", "DueDate", "ShipDate", "OnlineOrder", "CustomerID", "SalesPersonID",
        "TerritoryID", "CurrencyRateID", "SubTotal", "TaxAmt", "Freight", 
        "TotalDue", "ShipMethod"]]

In [7]:
currency_rate = currency_rate[["CurrencyRateID", "FromCurrencyCode", "ToCurrencyCode", "AverageRate"]]

In [8]:
orders = orders.merge(currency_rate, how = "left", on = "CurrencyRateID")

In [9]:
orders.drop(columns = {"FromCurrencyCode"}, inplace = True)

In [10]:
orders["ToCurrencyCode"] = orders["ToCurrencyCode"].fillna("USD")

In [11]:
orders = orders.rename(columns = {"ToCurrencyCode": "OriginalCurrency"})

In [12]:
orders["AverageRate"] = orders["AverageRate"].fillna(1)

In [13]:
order_details = order_details.merge(special_offer, how = "left", on = "SpecialOfferID")

In [14]:
order_details = order_details[["SalesOrderID", "OrderQty", "ProductID", "UnitPrice", "UnitPriceDiscount", "LineTotal", "Type"]]

In [15]:
order_details = order_details.rename(columns = {"Type": "DiscountType"})

In [16]:
order_details["GrossAmount"] = order_details["UnitPrice"] * order_details["OrderQty"]

In [17]:
order_details["DiscountAmount"] = (order_details["UnitPrice"] * order_details["UnitPriceDiscount"]) * order_details["OrderQty"]

In [18]:
orders = orders.merge(order_details,how = "left", on = "SalesOrderID")

In [20]:
orders["UnitPrice_USD"] = orders["UnitPrice"] * orders["AverageRate"]

orders["LineTotal_USD"] = orders["LineTotal"] * orders["AverageRate"]

orders["GrossAmount_USD"] = (
    orders["UnitPrice"] * orders["OrderQty"] * orders["AverageRate"]
)

orders["TaxAmt_USD"] = orders["TaxAmt"] * orders["AverageRate"]

orders["Freight_USD"] = orders["Freight"] * orders["AverageRate"]

orders["DiscountAmount_USD"] = orders["DiscountAmount"] * orders["AverageRate"]
"""
orders["OrderProductTotal_USD"] = (
    orders["LineTotal"] +
    orders["TaxAmt"] +
    orders["Freight"]
) * orders["AverageRate"]
"""
orders["OrderTotal_USD"] = orders["TotalDue"] * orders["AverageRate"]

In [21]:
orders = orders.drop(columns = {"SubTotal"})

In [29]:
orders = orders.rename(columns = {"AverageRate": "Rate", "TotalDue": "OrderTotal"})

In [30]:
from sqlalchemy import create_engine

engine = create_engine(
    "CONNECTION",
    fast_executemany=True
)

orders.to_sql(
    "sales",
    engine,
    if_exists="replace",
    schema="staging",
    index=False
)

-1

## DATE INFORMATIONS

In [4]:
start_date = "2010-01-01"
end_date = "2025-12-31"

date_range = pd.date_range(start=start_date, end=end_date)

date = pd.DataFrame({
    "Date": date_range,
})

date["Year"] = date["Date"].dt.year
date["Month"] = date["Date"].dt.month
date["MonthName"] = date["Date"].dt.strftime("%B")
date["Quarter"] = date["Date"].dt.quarter
date["Week"] = date["Date"].dt.isocalendar().week
date["Day"] = date["Date"].dt.day
date["DayName"] = date["Date"].dt.strftime("%A")

In [7]:
from sqlalchemy import create_engine

engine = create_engine(
    "CONNECTION",
    fast_executemany=True
)

date.to_sql(
    "dim_date",
    engine,
    if_exists="replace",
    schema="warehouse",
    index=False
)

-1